# LiITA SPARQL Query Explorer

Executes gold SPARQL queries from the test dataset against the LiITA endpoint and displays the result sets.

**Configure** the cell below, then run all cells.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# IDs to run — leave empty [] to run ALL test cases
QUERY_IDS = []

# Maximum rows to display per query (the query itself is never limited)
MAX_DISPLAY_ROWS = 20

# SPARQL endpoint
ENDPOINT = "https://liita.it/sparql"

# Timeout per query in seconds
TIMEOUT = 60

# Dataset path (None = auto-detect relative to this notebook)
DATASET_PATH = None

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown, HTML

# Resolve project root: notebook lives in <root>/notebooks/
# __file__ is not defined in Jupyter, so we derive root from cwd or a known relative path
try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Resolve dataset path
if DATASET_PATH is None:
    _candidates = [
        ROOT / "nl2sparql" / "data" / "test_dataset.json",
        ROOT / "data" / "test_dataset.json",
    ]
    DATASET_PATH = next((p for p in _candidates if p.exists()), _candidates[0])
else:
    DATASET_PATH = Path(DATASET_PATH)

print(f"Root     : {ROOT}")
print(f"Dataset  : {DATASET_PATH}  (exists: {DATASET_PATH.exists()})")
print(f"Endpoint : {ENDPOINT}")

In [ ]:
# Load test dataset and apply ID filter

with open(DATASET_PATH, encoding="utf-8") as f:
    dataset = json.load(f)

all_cases = dataset["test_cases"]

if QUERY_IDS:
    id_set = set(QUERY_IDS)
    selected = [tc for tc in all_cases if tc["id"] in id_set]
    missing = id_set - {tc["id"] for tc in selected}
    if missing:
        print(f"Warning: IDs not found in dataset: {sorted(missing)}")
else:
    selected = all_cases

print(f"Dataset  : {len(all_cases)} total test cases")
print(f"Selected : {len(selected)} queries to run")
if QUERY_IDS:
    print(f"IDs      : {sorted(tc['id'] for tc in selected)}")

In [ ]:
# SPARQL execution helper

def run_query(sparql: str, endpoint: str = ENDPOINT, timeout: int = TIMEOUT):
    """Execute a SPARQL query and return (variables, rows, error)."""
    try:
        from SPARQLWrapper import SPARQLWrapper, JSON, POST, URLENCODED
    except ImportError:
        return [], [], "SPARQLWrapper not installed — run: pip install SPARQLWrapper"

    try:
        client = SPARQLWrapper(endpoint)
        client.setQuery(sparql)
        client.setReturnFormat(JSON)
        client.setTimeout(timeout)
        client.setMethod(POST)
        client.setRequestMethod(URLENCODED)
        client.addCustomHttpHeader("Accept", "application/sparql-results+json")

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=RuntimeWarning, module="SPARQLWrapper")
            raw = client.query().convert()

        if isinstance(raw, bytes):
            raw = json.loads(raw.decode("utf-8", errors="replace"))

        variables = raw.get("head", {}).get("vars", [])
        bindings  = raw.get("results", {}).get("bindings", [])
        rows = [{var: b[var]["value"] for var in b} for b in bindings]
        return variables, rows, None

    except Exception as exc:
        msg = str(exc)
        if "timeout" in msg.lower():
            msg = f"Timeout after {timeout}s"
        return [], [], msg


def display_result(tc: dict, variables: list, rows: list, error: str | None) -> None:
    """Render one test case result as a formatted notebook section."""
    tid      = tc["id"]
    category = tc.get("category", "—")
    patterns = ", ".join(tc.get("patterns", []))
    nl_en    = tc.get("nl_en", "")
    sparql   = tc.get("sparql", "")

    # ── Header ────────────────────────────────────────────────────────────
    display(Markdown(
        f"---\n"
        f"### [{tid}] {nl_en}\n"
        f"**Category:** {category} &nbsp;|&nbsp; **Patterns:** {patterns or '—'}"
    ))

    # ── SPARQL query (collapsed details) ──────────────────────────────────
    display(HTML(
        "<details><summary><b>SPARQL query</b></summary>"
        f"<pre style='background:#f6f8fa;padding:10px;border-radius:4px;font-size:12px'>"
        f"{sparql.replace('<', '&lt;').replace('>', '&gt;')}"
        "</pre></details>"
    ))

    # ── Results ───────────────────────────────────────────────────────────
    if error:
        display(HTML(f"<p style='color:red'><b>Error:</b> {error}</p>"))
        return

    total = len(rows)
    shown = min(total, MAX_DISPLAY_ROWS)

    if total == 0:
        display(HTML("<p style='color:gray'><i>No results returned.</i></p>"))
        return

    caption = f"{total} result{'s' if total != 1 else ''}"
    if shown < total:
        caption += f" &nbsp;(showing first {shown})"

    display(HTML(f"<p><b>Results:</b> {caption}</p>"))

    df = pd.DataFrame(rows[:shown], columns=variables)
    # Truncate long URI values for readability
    for col in df.columns:
        df[col] = df[col].apply(
            lambda v: ("…" + str(v)[-60:]) if isinstance(v, str) and len(v) > 64 else v
        )
    display(df)

In [ ]:
# Run selected queries and display results

errors = []

for i, tc in enumerate(selected, 1):
    sparql = tc.get("sparql", "")
    if not sparql:
        display(Markdown(f"### [{tc['id']}] — no SPARQL query defined"))
        continue

    print(f"[{i}/{len(selected)}] Executing query {tc['id']}...", end=" ", flush=True)
    variables, rows, error = run_query(sparql)
    print(f"{len(rows)} rows" if not error else f"ERROR: {error}")

    display_result(tc, variables, rows, error)

    if error:
        errors.append((tc["id"], error))

# ── Final summary ─────────────────────────────────────────────────────────────
display(Markdown("---"))
print(f"Done. {len(selected)} queries executed, {len(errors)} error(s).")
if errors:
    print("\nErrors:")
    for eid, emsg in errors:
        print(f"  [{eid}] {emsg}")